<a href="https://colab.research.google.com/github/domore9630-hue/PINNs-Physics-AI-Solutions/blob/main/01_PINNs_Intro_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available (GPU): {torch.cuda.is_available()}")

PyTorch Version: 2.11.0+cpu
CUDA Available (GPU): False


In [ ]:
# بناء شبكة عصبية بسيطة تتكون من طبقة مدخلات وطبقتين خفيتين ومخرج واحد
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 1)
        )

    def forward(self, x):
        return self.net(x)

# تجربة الشبكة العصبية بمدخلات عشوائية
model = PINN()
test_input = torch.tensor([[0.5]])
test_output = model(test_input)

print("PINN Model Architecture:")
print(model)
print(f"\nTest Output for Input 0.5: {test_output.item():.4f}")

PINN Model Architecture:
PINN(
  (net): Sequential(
    (0): Linear(in_features=1, out_features=20, bias=True)
    (1): Tanh()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): Tanh()
    (4): Linear(in_features=20, out_features=1, bias=True)
  )
)

Test Output for Input 0.5: -0.3042


In [ ]:
# فكرة لحساب خسارة الفيزياء لـ معادلة تفاضلية: du/dt + u = 0
import torch

def physics_loss(model, t):
    t.requires_grad_(True)
    u = model(t)
    # حساب المشتقة du/dt تلقائياً
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]

    # بقايا المعادلة الفيزيائية (Residual)
    physics_residual = u_t + u

    # حساب خسارة الفيزياء (يجب أن تقترب من الصفر)
    return torch.mean(physics_residual**2)

In [ ]:

import torch
import torch.nn as nn

# 1. تعريف نموذج الشبكة العصبية
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 1)
        )

    def forward(self, x):
        return self.net(x)

# 2. إنشاء كائن من النموذج
model = PINN()

# 3. إعداد نقاط التدريب المكانية (Collocation Points)
# تمكين تتبع المشتقة عبر requires_grad_(True)
x_physics = torch.linspace(0, 2, 100).view(-1, 1).requires_grad_(True)

# 4. التمرير الأمامي للحصول على المخرج u
u = model(x_physics)

# 5. حساب المشتقة du/dx باستخدام التفاضل التلقائي (autograd)
du_dx = torch.autograd.grad(
    outputs=u,
    inputs=x_physics,
    grad_outputs=torch.ones_like(u),
    create_graph=True
)[0]

# 6. حساب متبقي المعادلة الفيزيائية (Physics Residual)
# المعاير الفيزيائي المطلوب: du/dx + u = 0
physics_residual = du_dx + u

# 7. حساب خسارة الفيزياء (يجب أن تقترب من الصفر أثناء التدريب)
loss_physics = torch.mean(physics_residual ** 2)

print("--- نتيجة حساب الفيزياء الجبرية ---")
print(f"قيمة خسارة الفيزياء العشوائية قبل التدريب: {loss_physics.item():.6f}")

--- نتيجة حساب الفيزياء الجبرية ---
قيمة خسارة الفيزياء العشوائية قبل التدريب: 0.035650


In [2]:
import torch
import torch.nn as nn

# 1. إعداد المُحسن (Optimizer) ونسبة التعلم (Learning Rate)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 2. الشروط الابتدائية: عند x = 0 يجب أن تكون u = 1
x_ic = torch.tensor([[0.0]], requires_grad=True)
u_ic_target = torch.tensor([[1.0]])

print("بدء تدريب نموذج PINN...\n")

# 3. حلقة التدريب (Training Loop)
epochs = 1000
for epoch in range(1, epochs + 1):
    optimizer.zero_grad()

    # أ) خسارة الشرط الابتدائي (Boundary/Initial Condition Loss)
    u_ic_pred = model(x_ic)
    loss_ic = torch.mean((u_ic_pred - u_ic_target) ** 2)

    # ب) خسارة الفيزياء عبر النطاق (Physics Loss)
    x_physics = torch.linspace(0, 2, 100).view(-1, 1).requires_grad_(True)
    u_physics = model(x_physics)

    du_dx = torch.autograd.grad(
        outputs=u_physics,
        inputs=x_physics,
        grad_outputs=torch.ones_like(u_physics),
        create_graph=True
    )[0]

    loss_physics = torch.mean((du_dx + u_physics) ** 2)

    # ج) الخسارة الكلية (Total Loss)
    total_loss = loss_ic + loss_physics

    # د) التمرير الخلفي وتحديث الأوزان (Backpropagation)
    total_loss.backward()
    optimizer.step()

    # طباعة التقدم كل 200 دورة
    if epoch % 200 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d}/{epochs} | Total Loss: {total_loss.item():.6f} | Physics Loss: {loss_physics.item():.6f}")

print("\nتم اكتمال تدريب النموذج بنجاح!")

بدء تدريب نموذج PINN...

Epoch    1/1000 | Total Loss: 1.179890 | Physics Loss: 0.035650
Epoch  200/1000 | Total Loss: 0.000042 | Physics Loss: 0.000042
Epoch  400/1000 | Total Loss: 0.000006 | Physics Loss: 0.000006
Epoch  600/1000 | Total Loss: 0.000005 | Physics Loss: 0.000005
Epoch  800/1000 | Total Loss: 0.000004 | Physics Loss: 0.000004
Epoch 1000/1000 | Total Loss: 0.000004 | Physics Loss: 0.000004

تم اكتمال تدريب النموذج بنجاح!
